# Processing Host Plant Data for Machine Learning Models
## Erica Keklak | 2025-06-25

In [4]:
# Import necessary libraries and packages

import pandas as pd
import geopandas as gpd

## Purpose and Seasonality Data

In [6]:
# Host plants sorted by purpose

host_purposes = pd.read_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Host Ecological/Host Plant Sorting.csv',
                            dtype = {0: str, 1: str, 2: str, 3: str, 4: str, 5: str, 6: str, 7: str, 8: str, 9: str, 10: str, 11: str, 12: str}
                           )
host_purposes = host_purposes.drop(columns = ['Rank', 'U.S. Native Range', 'U.S. Intro Range', 'U.S. Invasive Report Range',
                                              'U.S. Invasive List or Law Range', 'Fungal Vulnerability?', 'Human Usage Description'
                                             ], axis = 1)
host_purposes = host_purposes.dropna(subset = ['Used for Food?', 'Used for Timber or Paper?', 'Used as Ornamental?'], axis = 0)
host_purposes

,Scientific Name,Popular Common Name,SLF Life Stage Support,Growth Habit,Used for Food?,Used for Timber or Paper?,Used as Ornamental?
0,Acacia sp.,Acacias/Wattles,-,All,Yes,Yes,Yes
1,Acer buergerianum,Trident maple,-,Tree,No,No,Yes
2,Acer negundo,Boxelder,"Egg, Nymph",Tree,No,No,Yes
3,Acer palmatum,Japanese maple,-,Tree,No,No,Yes
4,Acer pictum subsp. Mono (Maxim.),Painted maple (subsp. Of Mono maple),-,Tree,No,No,Yes
...,...,...,...,...,...,...,...
163,Vaccinium angustifolium,Lowbush blueberry,Nymph,Shrub,Yes,No,No
164,Viburnum prunifolium,Blackhaw,Egg,Shrub,Yes,No,Yes
168,Vitis sp.,Wild grape,"Nymph, Adult",Vine,Yes,No,Yes
170,Zanthoxylum simulans,Chinese pepper,Adult,Tree,No,No,Yes


In [8]:
# Set up the list of species of concern for purpose-specific models and for life stage support

food_species = []
fiber_species = []
ornamental_species = []
for i in range(0, len(host_purposes)):
    if host_purposes.iloc[i].loc['Used for Food?'] == 'Yes':
        food_species = food_species + [host_purposes.iloc[i].loc['Scientific Name']]
    if host_purposes.iloc[i].loc['Used for Timber or Paper?'] == 'Yes':
        fiber_species = fiber_species + [host_purposes.iloc[i].loc['Scientific Name']]
    if host_purposes.iloc[i].loc['Used as Ornamental?'] == 'Yes':
        ornamental_species = ornamental_species + [host_purposes.iloc[i].loc['Scientific Name']]

egg_species = []
nymph_species = []
adult_species = []
for i in range(0, len(host_purposes)):
    life_stages_string = host_purposes.iloc[i].loc['SLF Life Stage Support']
    if 'Egg' not in life_stages_string and 'Nymph' not in life_stages_string and 'Adult' not in life_stages_string:
        egg_species = egg_species + [host_purposes.iloc[i].loc['Scientific Name']]
        nymph_species = nymph_species + [host_purposes.iloc[i].loc['Scientific Name']]
        adult_species = adult_species + [host_purposes.iloc[i].loc['Scientific Name']]
    if 'Egg' in life_stages_string:
        egg_species = egg_species + [host_purposes.iloc[i].loc['Scientific Name']]
    if 'Nymph' in life_stages_string:
        nymph_species = nymph_species + [host_purposes.iloc[i].loc['Scientific Name']]
    if 'Adult' in life_stages_string:
        adult_species = adult_species + [host_purposes.iloc[i].loc['Scientific Name']]

## Observation Data

In [16]:
# Read all the host plant data from iNaturalist:

begin_path = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Host Ecological/'
end_path = ' iNaturalist.csv'
def makepath(path: str, b = begin_path, e = end_path) -> str:
    return str(b + path + e)

acacia_sp_iNat = pd.read_csv(makepath('Acacia sp Wattles/Acacia sp'))
acer_buergerianum_iNat = pd.read_csv(makepath('Acer buergerianum Trident maple/Acer buergerianum'))
acer_negundo_iNat = pd.read_csv(makepath('Acer negundo Boxelder/Acer negundo'), dtype = {15: object})
acer_palmatum_iNat = pd.read_csv(makepath('Acer palmatum Japanese maple/Acer palmatum'))
acer_pictum_iNat = pd.read_csv(makepath('Acer pictum Mono maple/Acer pictum'))
acer_platanoides_iNat = pd.read_csv(makepath('Acer platanoides Norway maple/Acer platanoides'), dtype = {15: object})
acer_pseudoplatanus_iNat = pd.read_csv(makepath('Acer pseudoplatanus Sycamore maple/Acer pseudoplatanus'))
acer_rubrum_iNat = pd.read_csv(makepath('Acer rubrum Red maple/Acer rubrum'), dtype = {15: object})
acer_saccharinum_iNat = pd.read_csv(makepath('Acer saccharinum Silver maple/Acer saccharinum'))
acer_saccharum_iNat = pd.read_csv(makepath('Acer saccharum Sugar maple/Acer saccharum'), dtype = {15: object})
actinidia_chinensis_iNat = pd.read_csv(makepath('Actinidia chinensis Kiwi/Actinidia chinensis'))
ailanthus_altissima_iNat = pd.read_csv(makepath('Ailanthus altissima Tree of heaven/Ailanthus altissima'))
albizia_julibrissin_iNat = pd.read_csv(makepath('Albizia julibrissin Persian silk tree/Albizia julibrissin'))
alcea_sp_iNat = pd.read_csv(makepath('Alcea sp Hollyhocks/Alcea sp'))
alnus_incana_iNat = pd.read_csv(makepath('Alnus incana Gray alder/Alnus incana'))
amelanchier_sp_iNat = pd.read_csv(makepath('Amelanchier sp Serviceberries/Amelanchier sp'))
# angelica_dahurica
aralia_cordata_iNat = pd.read_csv(makepath('Aralia cordata Spikenard/Aralia cordata'))
aralia_elata_iNat = pd.read_csv(makepath('Aralia elata Japanese angelica/Aralia elata'))
arctium_lappa_iNat = pd.read_csv(makepath('Arctium lappa Greater burdock/Arctium lappa'))
armoracia_rusticana_iNat = pd.read_csv(makepath('Armoracia rusticana Horseradish/Armoracia rusticana'))
betula_alleghaniensis_iNat = pd.read_csv(makepath('Betula alleghaniensis Yellow birch/betula alleghaniensis'))
betula_lenta_iNat = pd.read_csv(makepath('Betula lenta Sweet birch/Betula lenta'))
betula_nigra_iNat = pd.read_csv(makepath('Betula nigra River birch/Betula nigra'))
betula_papyrifera_iNat = pd.read_csv(makepath('Betula papyrifera Paper birch/Betula papyrifera'))
betula_pendula_iNat = pd.read_csv(makepath('Betula pendula European white birch/Betula pendula'))
betula_platyphylla = pd.read_csv(makepath('Betula platyphylla Asian white birch/Betula platyphylla'))
broussonetia_papyrifera_iNat = pd.read_csv(makepath('Broussonetia papyrifera Paper mulberry/Broussonetia papyrifera'))
buxus_microphylla_iNat = pd.read_csv(makepath('Buxus microphylla Small-leaved box/Buxus microphylla'))
# buxus_sinica_iNat
callistephus_chinensis_iNat = pd.read_csv(makepath('Callistephus chinensis Chinese aster/Callistephus chinensis'))
camellia_sinensis_iNat = pd.read_csv(makepath('Camellia sinensis Tea/Camellia sinensis'))
cannabis_sativa_iNat = pd.read_csv(makepath('Cannabis sativa Hemp/Cannabis sativa'))
carpinus_caroliniana_iNat = pd.read_csv(makepath('Carpinus caroliniana American hornbeam/Carpinus caroliniana'))
carya_glabra_iNat = pd.read_csv(makepath('Carya glabra Pignut hickory/Carya glabra'))
carya_ovata_iNat = pd.read_csv(makepath('Carya ovata Shagbark hickory/Carya ovata'))
castanea_crenata_iNat = pd.read_csv(makepath('Castanea crenata Japanese chestnut/Castanea crenata'))
catalpa_bungei_iNat = pd.read_csv(makepath('Catalpa bungei Manchurian catalpa/Catalpa bungei'))
# cedrela_fissilis
# celastrus_orbiculatus
celastrus_orbiculatus_iNat = pd.read_csv(makepath('Celastrus orbiculatus Oriental bittersweet/Celastrus orbiculatus'))
chamerion_angustifolium_iNat = pd.read_csv(makepath('Chamerion angustifolium Fireweed/Chamerion angustifolium'))
colutea_arborescens_iNat = pd.read_csv(makepath('Colutea arborescens Bladder senna/Colutea arborescens'))
cornus_sp_iNat = pd.read_csv(makepath('Cornus sp Dogwoods/Cornus sp'), dtype = {15: object, 40: object})
corylus_americana_iNat = pd.read_csv(makepath('Corylus americana American hazelnut/Corylus americana'))
diospyros_kaki_iNat = pd.read_csv(makepath('Diospyros kaki Japanese persimmon/Diospyros kaki'))
elaeagnus_umbellata_iNat = pd.read_csv(makepath('Elaeagnus umbellata Autumn olive/Elaeagnus umbellata'))
euphorbia_pulcherrima_iNat = pd.read_csv(makepath('Euphorbia pulcherrima Poinsettia/Euphorbia pulcherrima'))
fagus_grandifolia_iNat = pd.read_csv(makepath('Fagus grandifolia American beech/Fagus grandifolia'), dtype = {15: object, 41: object})
ficus_carica_iNat = pd.read_csv(makepath('Ficus carica Common fig/Ficus carica'))
firmiana_simplex_iNat = pd.read_csv(makepath('Firmiana simplex Chinese parasol tree/Firmiana simplex'))
forsythia_sp_iNat = pd.read_csv(makepath('Forsythia sp Forsythias/Forsythia sp'))
fraxinus_sp_iNat = pd.read_csv(makepath('Fraxinus sp Ashes/Fraxinus sp'))
glycine_max_iNat = pd.read_csv(makepath('Glycine max Soybean/Glycine max'))
hibiscus_sp_iNat = pd.read_csv(makepath('Hibiscus sp Hibiscuses/Hibiscus sp'), dtype = {40: object})
humulus_japonicus_iNat = pd.read_csv(makepath('Humulus japonicus Japanese hops/Humulus japonicus'))
humulus_lupulus_iNat = pd.read_csv(makepath('Humulus lupulus Common hops/Humulus lupulus'))
juglans_sp_iNat = pd.read_csv(makepath('Juglans sp Walnuts/Juglans sp'))
juniperus_chinensis_iNat = pd.read_csv(makepath('Juniperus chinensis Chinese juniper/Juniperus chinensis'))
ligustrum_lucidum_iNat = pd.read_csv(makepath('Ligustrum lucidum Glossy privet/Ligustrum lucidum'))
lindera_benzoin_iNat = pd.read_csv(makepath('Lindera benzoin Northern spicebush/Lindera benzoin'))
liriodendron_tulipifera_iNat = pd.read_csv(makepath('Liriodendron tulipifera Tulip tree/Liriodendron tulipifera'), dtype = {15: object})
lonicera_sp_iNat = pd.read_csv(makepath('Lonicera sp Honeysuckles/Lonicera sp'))
luffa_sp_iNat = pd.read_csv(makepath('Luffa sp Sponge gourds/Luffa sp'))
maackia_amurensis_iNat = pd.read_csv(makepath('Maackia amurensis Amur maackia/Maackia amurensis'))
magnolia_kobus_iNat = pd.read_csv(makepath('Magnolia kobus Kobus magnolia/Magnolia kobus'))
magnolia_obovata_iNat = pd.read_csv(makepath('Magnolia obovata Japanese bigleaf magnolia/Magnolia obovata'))
mallotus_japonicus_iNat = pd.read_csv(makepath('Mallotus japonicus Mallotus/Mallotus japonicus'))
malus_sp_iNat = pd.read_csv(makepath('Malus sp Apples/Malus sp'))
melia_azedarach_iNat = pd.read_csv(makepath('Melia azedarach Chinaberry/Melia azedarach'))
metaplexis_japonica_iNat = pd.read_csv(makepath('Metaplexis japonica Rough potato/Metaplexis japonica'))
monarda_sp_iNat = pd.read_csv(makepath('Monarda sp Beebalms and bergamots/Monarda sp'), dtype = {40: object})
morus_alba_iNat = pd.read_csv(makepath('Morus alba White mulberry/Morus alba'))
morus_bombycis_iNat = pd.read_csv(makepath('Morus bombycis Korean mulberry/Morus bombycis'))
nicotiana_sp_iNat = pd.read_csv(makepath('Nicotiana sp Tobacco/Nicotiana sp'))
nyssa_sylvatica_iNat = pd.read_csv(makepath('Nyssa sylvatica Black tupelo/Nyssa sylvatica'))
ocimum_basilicum_iNat = pd.read_csv(makepath('Ocimum basilicum Basil/Ocimum basilicum'))
osmanthus_sp_iNat = pd.read_csv(makepath('Osmanthus sp Devilwoods/Osmanthus sp'))
ostrya_virginiana_iNat = pd.read_csv(makepath('Ostrya virginiana American hophornbeam/Ostrya virginiana'))
parthenocissus_quinquefolia_iNat = pd.read_csv(makepath('Parthenocissus quinquefolia Virginia creeper/Parthenocissus quinquefolia'))
paulownia_kawakamii_iNat = pd.read_csv(makepath('Paulownia kawakamii Sapphire dragon tree/Paulownia kawakamii'))
paulownia_tomentosa_iNat = pd.read_csv(makepath('Paulownia tomentosa Princess tree/Paulownia tomentosa'))
phellodendron_amurense_iNat = pd.read_csv(makepath('Phellodendron amurense Amur corktree/Phellodendron amurense'))
# philadelphus_schrenkii
phyllostachys_heterocycla_iNat = pd.read_csv(makepath('Phyllostachys heterocycla Moso bamboo/Phyllostachys heterocycla'))
# picrasma_quassioides
pinus_strobus_iNat = pd.read_csv(makepath('Pinus strobus Eastern white pine/Pinus strobus'), dtype = {15: object})
platanus_occidentalis_iNat = pd.read_csv(makepath('Platanus occidentalis American sycamore/Platanus occidentalis'))
platanus_orientalis_iNat = pd.read_csv(makepath('Platanus orientalis Oriental plane/Platanus orientalis'))
platanus_x_acerifolia_iNat = pd.read_csv(makepath('Platanus x acerifolia London plane/Platanus x acerifolia'))
platycarya_strobilacea_iNat = pd.read_csv(makepath('Platycarya strobilacea Platycarya/Platycarya strobilacea'))
platycladus_orientalis_iNat = pd.read_csv(makepath('Platycladus orientalis Chinese arborvitae/Platycladus orientalis'))
populus_alba_iNat = pd.read_csv(makepath('Populus alba White poplar/Populus alba'))
populus_grandidentata_iNat = pd.read_csv(makepath('Populus grandidentata Bigtooth aspen/Populus grandidentata'))
# populus_koreana_iNat
populus_simonii_iNat = pd.read_csv(makepath('Populus simonii Simon\'s poplar/Populus simonii'))
# populus_tomentiglandulosa
populus_tomentosa_iNat = pd.read_csv(makepath('Populus tomentosa Chinese white poplar/Populus tomentosa'))
prunus_armeniaca_iNat = pd.read_csv(makepath('Prunus armeniaca Apricot/Prunus armeniaca'))
prunus_avium_iNat = pd.read_csv(makepath('Prunus avium Wild cherry/Prunus avium'))
prunus_cerasus_iNat = pd.read_csv(makepath('Prunus cerasus Sour cherry/Prunus cerasus'))
prunus_mume_iNat = pd.read_csv(makepath('Prunus mume Flowering apricot/Prunus mume'))
prunus_persica_iNat = pd.read_csv(makepath('Prunus persica Peach/Prunus persica'))
prunus_salicina_iNat = pd.read_csv(makepath('Prunus salicina Chinese plum/Prunus salicina'))
prunus_serotina_iNat = pd.read_csv(makepath('Prunus serotina Black cherry/Prunus serotina'), dtype = {15: object})
prunus_serrulata_iNat = pd.read_csv(makepath('Prunus serrulata Japanese cherry/Prunus serrulata'))
prunus_x_yedoensis_iNat = pd.read_csv(makepath('Prunus x yedoensis Yoshino cherry/Prunus x yedoensis'))
pseudocydonia_sinensis_iNat = pd.read_csv(makepath('Pseudocydonia sinensis Chinese quince/Pseudocydonia sinensis'))
pterocarya_stenoptera_iNat = pd.read_csv(makepath('Pterocarya stenoptera Chinese wingnut/Pterocarya stenoptera'))
punica_granatum_iNat = pd.read_csv(makepath('Punica granatum Pomegranate/Punica granatum'))
pyrus_sp_iNat = pd.read_csv(makepath('Pyrus sp Pears/Pyrus sp'))
quercus_sp_iNat = pd.read_csv(makepath('Quercus sp Oaks/Quercus sp iNaturalist Part 1.csv', e = ''), dtype = {15: object, 41: object})
quercus_2 = pd.read_csv(makepath('Quercus sp Oaks/Quercus sp iNaturalist Part 2.csv', e = ''), dtype = {15: object, 41: object})
quercus_3 = pd.read_csv(makepath('Quercus sp Oaks/Quercus sp iNaturalist Part 3.csv', e = ''), dtype = {15: object, 41: object})
quercus_sp_iNat = quercus_sp_iNat.merge(quercus_2, how = 'outer').merge(quercus_3, how = 'outer')
rhus_chinensis_iNat = pd.read_csv(makepath('Rhus chinensis Chinese sumac/Rhus chinensis'))
rhus_typhina_iNat = pd.read_csv(makepath('Rhus typhina Staghorn sumac/Rhus typhina'))
robinia_pseudoacacia_iNat = pd.read_csv(makepath('Robinia pseudoacacia Black locust/Robinia pseudoacacia'))
rosa_sp_iNat = pd.read_csv(makepath('Rosa sp Roses/Rosa sp'), dtype = {15: object})
rubus_sp_iNat = pd.read_csv(makepath('Rubus sp Brambles/Rubus sp'), dtype = {15: object})
salix_sp_iNat = pd.read_csv(makepath('Salix sp Willows/Salix sp'), dtype = {15: object, 40: object})
salvia_sp_iNat = pd.read_csv(makepath('Salvia sp Sages/Salvia sp'))
sassafras_albidum_iNat = pd.read_csv(makepath('Sassafras albidum Sassafras/Sassafras albidum'), dtype = {15: object})
sorbaria_sorbifolia_iNat = pd.read_csv(makepath('Sorbaria sorbifolia False spiraea/Sorbaria sorbifolia'))
sorbus_commixta_iNat = pd.read_csv(makepath('Sorbus commixta Japanese rowan/Sorbus commixta'))
styphnolobium_japonicum_iNat = pd.read_csv(makepath('Styphnolobium japonicum Japanese pagoda tree/Styphnolobium japonicum'))
styrax_japonicus_iNat = pd.read_csv(makepath('Styrax japonicus Japanese snowbell/Styrax japonicus'))
styrax_obassia_iNat = pd.read_csv(makepath('Styrax obassia Fragrant snowbell/Styrax obassia'))
syringa_vulgaris_iNat = pd.read_csv(makepath('Syringa vulgaris Common lilac/Syringa vulgaris'))
tamarix_chinensis_iNat = pd.read_csv(makepath('Tamarix chinensis Five-stamen tamarisk/Tamarix chinensis'))
tetradium_sp_iNat = pd.read_csv(makepath('Tetradium sp Tetradium/Tetradium sp'))
thuja_occidentalis_iNat = pd.read_csv(makepath('Thuja occidentalis Arborvitae/Thuja occidentalis'))
tilia_americana_iNat = pd.read_csv(makepath('Tilia americana American basswood/Tilia americana'))
toona_sinensis_iNat = pd.read_csv(makepath('Toona sinensis Chinese mahogany/Toona sinensis'))
toxicodendron_radicans_iNat = pd.read_csv(makepath('Toxicodendron radicans Eastern poison ivy/Toxicodendron radicans'))
# toxicodendron_vernicifuum
ulmus_sp_iNat = pd.read_csv(makepath('Ulmus sp Elms/Ulmus sp'))
vaccinium_angustifolium_iNat = pd.read_csv(makepath('Vaccinium angustifolium Lowbush blueberry/Vaccinium angustifolium'))
viburnum_prunifolium_iNat = pd.read_csv(makepath('Viburnum prunifolium Blackhaw/Viburnum prunifolium'))
vitis_sp_iNat = pd.read_csv(makepath('Vitis sp Grapes/Vitis sp'))
# add winegrape?
zanthoxylum_simulans_iNat = pd.read_csv(makepath('Zanthoxylum simulans Chinese pepper/Zanthoxylum simulans'))
zelkova_serrata_iNat = pd.read_csv(makepath('Zelkova serrata Japanese zelkova/Zelkova serrata'))

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Host Ecological/Tamarix chinensis Five-stamen tamarisk/Tamarix chinensis iNaturalist.csv'

In [ ]:
hosts_iNat = [acacia_sp_iNat, acer_buergerianum_iNat, acer_negundo_iNat, acer_palmatum_iNat, acer_pictum_iNat, acer_platanoides_iNat,
              acer_pseudoplatanus_iNat, acer_rubrum_iNat, acer_saccharinum_iNat, acer_saccharum_iNat, actinidia_chinensis_iNat, ailanthus_altissima_iNat,
              albizia_julibrissin_iNat, alcea_sp_iNat, alnus_incana_iNat, amelanchier_sp_iNat, aralia_cordata_iNat, aralia_elata_iNat, arctium_lappa_iNat,
              armoracia_rusticana_iNat, betula_alleghaniensis_iNat, betula_lenta_iNat, betula_nigra_iNat, betula_papyrifera_iNat, betula_pendula_iNat,
              betula_platyphylla, broussonetia_papyrifera_iNat, buxus_microphylla_iNat, callistephus_chinensis_iNat, camellia_sinensis_iNat,
              cannabis_sativa_iNat, carpinus_caroliniana_iNat, carya_glabra_iNat, carya_ovata_iNat, castanea_crenata_iNat, catalpa_bungei_iNat,
              celastrus_orbiculatus_iNat, chamerion_angustifolium_iNat, colutea_arborescens_iNat, cornus_sp_iNat, corylus_americana_iNat,
              diospyros_kaki_iNat, elaeagnus_umbellata_iNat, euphorbia_pulcherrima_iNat, fagus_grandifolia_iNat, ficus_carica_iNat, firmiana_simplex_iNat,
              forsythia_sp_iNat, fraxinus_sp_iNat, glycine_max_iNat, hibiscus_sp_iNat, humulus_japonicus_iNat, humulus_lupulus_iNat, juglans_sp_iNat,
              juniperus_chinensis_iNat, ligustrum_lucidum_iNat, lindera_benzoin_iNat, liriodendron_tulipifera_iNat, lonicera_sp_iNat, luffa_sp_iNat,
              maackia_amurensis_iNat, magnolia_kobus_iNat, magnolia_obovata_iNat, mallotus_japonicus_iNat, malus_sp_iNat, melia_azedarach_iNat,
              metaplexis_japonica_iNat, monarda_sp_iNat, morus_alba_iNat, morus_bombycis_iNat, nicotiana_sp_iNat, nyssa_sylvatica_iNat,
              ocimum_basilicum_iNat, osmanthus_sp_iNat, ostrya_virginiana_iNat, parthenocissus_quinquefolia_iNat, paulownia_kawakamii_iNat,
              paulownia_tomentosa_iNat, phellodendron_amurense_iNat, phyllostachys_heterocycla_iNat, pinus_strobus_iNat, platanus_occidentalis_iNat,
              platanus_orientalis_iNat, platanus_x_acerifolia_iNat, platycarya_strobilacea_iNat, platycladus_orientalis_iNat, populus_alba_iNat,
              populus_grandidentata_iNat, populus_simonii_iNat, populus_tomentosa_iNat, prunus_armeniaca_iNat, prunus_avium_iNat, prunus_cerasus_iNat,
              prunus_mume_iNat, prunus_persica_iNat, prunus_salicina_iNat, prunus_serotina_iNat, prunus_serrulata_iNat, prunus_x_yedoensis_iNat,
              pseudocydonia_sinensis_iNat, pterocarya_stenoptera_iNat, punica_granatum_iNat, pyrus_sp_iNat, quercus_sp_iNat, rhus_chinensis_iNat,
              rhus_typhina_iNat, robinia_pseudoacacia_iNat, rosa_sp_iNat, rubus_sp_iNat, salix_sp_iNat, salvia_sp_iNat, sassafras_albidum_iNat,
              sorbaria_sorbifolia_iNat, sorbus_commixta_iNat, styphnolobium_japonicum_iNat, styrax_japonicus_iNat, styrax_obassia_iNat,
              syringa_vulgaris_iNat, tamarix_chinensis_iNat, tetradium_sp_iNat, thuja_occidentalis_iNat, tilia_americana_iNat, toona_sinensis_iNat,
              toxicodendron_radicans_iNat, ulmus_sp_iNat, vaccinium_angustifolium_iNat, viburnum_prunifolium_iNat, vitis_sp_iNat,
              zanthoxylum_simulans_iNat, zelkova_serrata_iNat
             ]

In [ ]:
hosts = {'acacia_sp': [acacia_sp_iNat, acacia_sp_usda],
         'acer_buergerianum': [acer_buergerianum_iNat],
         'acer_negundo': [acer_negundo_iNat],
         'acer_palmatum': [acer_palmatum_iNat],
         'acer_pictum': [acer_pictum_iNat],
         'acer_platanoides': [acer_platanoides_iNat],
         'acer_pseudoplatanus': [acer_pseudoplatanus_iNat],
         'acer_rubrum': [acer_rubrum_iNat],
         'acer_saccharinum': [acer_saccharinum_iNat],
         'acer_saccharum': [acer_saccharum_iNat],
         'actinidia_chinensis': [actinidia_chinensis_iNat],
         'ailanthus_altissima': [ailanthus_altissima_iNat],
         'albizia_julibrissin': [albizia_julibrissin_iNat],
         'acea_sp': [alcea_sp_iNat],
         'alnus_incana': [alnus_incana_iNat],
         'amelanchier_sp': [amelanchier_sp_iNat],
         'angelica_dahuria': [],
         'aralia_cordata': [aralia_cordata_iNat],
         'aralia_elata': [aralia_elata_iNat],
         'arctium_lappa': [arctium_lappa_iNat],
         'armoracia_rusticana': [armoracia_rusticana_iNat],
         'betula_alleghaniensis': [betula_alleghaniensis_iNat],
         'betula_lenta': [betula_lenta_iNat],
         'betula_nigra': [betula_nigra_iNat],
         'betula_papyrifera': [betula_papyrifera_iNat],
         'betula_pendula': [betula_pendula_iNat],
         'betula_platyphylla': [betula_platyphylla],
         'broussonetia_papyrifera': [broussonetia_papyrifera_iNat],
         'buxus_microphylla': [buxus_microphylla_iNat],
         'buxus_sinica': [],
         'callistephus_chinensis': [callistephus_chinensis_iNat],
         'camellia_sinensis': [camellia_sinensis_iNat],
         'cannabis_sativa': [cannabis_sativa_iNat],
         'carpinus_caroliniana': [carpinus_caroliniana_iNat],
         'carya_glabra': [carya_glabra_iNat],
         'carya_ovata': [carya_ovata_iNat],
         'castanea_crenata': [castanea_crenata_iNat],
         'catalpa_bungei': [catalpa_bungei_iNat],
         'cedrela_fissilis': [],
         'celastrus_orbiculatus': [celastrus_orbiculatus_iNat], # exists?
         'chamerion_angustifolium': [chamerion_angustifolium_iNat],
         'colutea_arborescens': [colutea_arborescens_iNat],
         'cornus_sp': [cornus_sp_iNat],
         'corylus_americana': [corylus_americana_iNat],
         'diospyros_kaki': [diospyros_kaki_iNat],
         'elaeagnus_umbellata': [elaeagnus_umbellata_iNat],
         'euphorbia_pulcherrima': [euphorbia_pulcherrima_iNat],
         'fagus_grandifolia': [fagus_grandifolia_iNat],
         'ficus_carica': [ficus_carica_iNat],
         'firmiana_simplex': [firmiana_simplex_iNat],
         'forsythia_sp': [forsythia_sp_iNat],
         'fraxinus_sp': [fraxinus_sp_iNat], 
         'glycine_max': [glycine_max_iNat],
         'hibiscus_sp': [hibiscus_sp_iNat],
         'humulus_japonicus': [humulus_japonicus_iNat],
         'humulus_lupulus': [humulus_lupulus_iNat],
         'juglans_sp': [juglans_sp_iNat],
         'juniperus_chinensis': [juniperus_chinensis_iNat],
         'ligustrum_lucidum': [ligustrum_lucidum_iNat],
         'lindera_benzoin': [lindera_benzoin_iNat],
         'liriodendron_tulipifera': [liriodendron_tulipifera_iNat],
         'lonicera_sp': [lonicera_sp_iNat],
         'luffa_sp': [luffa_sp_iNat],
         'maackia_amurensis': [maackia_amurensis_iNat],
         'magnolia_kobus': [magnolia_kobus_iNat],
         'magnolia_obovata': [magnolia_obovata_iNat],
         'mallotus_japonicus': [mallotus_japonicus_iNat],
         'malus_sp': [malus_sp_iNat],
         'melia_azedarach': [melia_azedarach_iNat],
         'meteplexis_japonica': [metaplexis_japonica_iNat],
         'monarda_sp': [monarda_sp_iNat],
         'morus_alba': [morus_alba_iNat],
         'morus_bombycis': [morus_bombycis_iNat],
         'nicotiana_sp': [nicotiana_sp_iNat],
         'nyssa_sylvatica': [nyssa_sylvatica_iNat],
         'ocimum_basilicum': [ocimum_basilicum_iNat],
         'osmanthus_sp': [osmanthus_sp_iNat],
         'ostrya_virginiana': [ostrya_virginiana_iNat],
         'pathenocissus_quinquefolia': [parthenocissus_quinquefolia_iNat],
         'paulownia_kawakamii': [paulownia_kawakamii_iNat],
         'paulownia_tomentosa': [paulownia_tomentosa_iNat],
         'phellodendron_amurense': [phellodendron_amurense_iNat],
         'philadelphus_schrenkii': [],
         'phyllostachys_heterocycla': [phyllostachys_heterocycla_iNat], 
         'picrasma_quassioides': [],
         'pinus_strobus': [pinus_strobus_iNat],
         'platanus_occidentalis': [platanus_occidentalis_iNat],
         'platanus_orientalis': [platanus_orientalis_iNat],
         'platanus_x_acerifolia': [platanus_x_acerifolia_iNat],
         'platycarya_strobilacea': [platycarya_strobilacea_iNat],
         'platycladus_orientalis': [platycladus_orientalis_iNat],
         'populus_alba': [populus_alba_iNat],
         'populus_grandidentata': [populus_grandidentata_iNat],
         'populus_koreana': [],
         'populus_simonii': [populus_simonii_iNat],
         'populus_tomentiglandulosa': [],
         'populus_tomentosa': [populus_tomentosa_iNat],
         'prunus_armeiaca': [prunus_armeniaca_iNat],
         'prunus_avium': [prunus_avium_iNat],
         'prunus_cerasus': [prunus_cerasus_iNat],
         'prunus_mume': [prunus_mume_iNat],
         'prunus_persica': [prunus_persica_iNat],
         'prunus_salicina': [prunus_salicina_iNat],
         'prunus_serotina': [prunus_serotina_iNat],
         'prunus_seruulata': [prunus_serrulata_iNat],
         'prunus_x_yedoensis': [prunus_x_yedoensis_iNat],
         'pseudocydonia_sinensis': [pseudocydonia_sinensis_iNat],
         'pseudocarya_stenoptera': [pterocarya_stenoptera_iNat],
         'punica_granatum': [punica_granatum_iNat],
         'pyrus_sp': [pyrus_sp_iNat],
         'quercus_sp': [quercus_sp_iNat],
         'rhus_chinensis': [rhus_chinensis_iNat],
         'rhus_typhina': [rhus_typhina_iNat],
         'robinia_pseudoacacia': [robinia_pseudoacacia_iNat],
         'rosa_sp': [rosa_sp_iNat],
         'rubus_sp': [rubus_sp_iNat],
         'salix_sp': [salix_sp_iNat],
         'salvia_sp': [salvia_sp_iNat],
         'sassafras_albidum': [sassafras_albidum_iNat],
         'sorbaria_sorbifolia': [sorbaria_sorbifolia_iNat],
         'sorbus_commixta': [sorbus_commixta_iNat],
         'styphnolobium_japonicum': [styphnolobium_japonicum_iNat],
         'styrax_japonicus': [styrax_japonicus_iNat],
         'styrax_obassia': [styrax_obassia_iNat],
         'syringa_vulgaris': [syringa_vulgaris_iNat],
         'tamarix_chinensis': [tamarix_chinensis_iNat],
         'tetradium_sp': [tetradium_sp_iNat],
         'thuja_occidentalis': [thuja_occidentalis_iNat],
         'tilia_americana': [tilia_americana_iNat],
         'toona_sinensis': [toona_sinensis_iNat],
         'toxicodendron_radicans': [toxicodendron_radicans_iNat],
         'toxicodendron_vernicifluum': [],
         'ulmus_sp': [ulmus_sp_iNat],
         'vaccinium_angustifolium': [vaccinium_angustifolium_iNat],
         'viburnum_prunifolium': [viburnum_prunifolium_iNat],
         'vitis_sp': [vitis_sp_iNat],
         'zanthoxylum_simulans': [zanthoxylum_simulans_iNat],
         'zelkova_serrata': [zelkova_serrata_iNat]
        }

In [ ]:
# Read the columns of each iNaturalist DataFrame and only select the ones that have bearing on this research

column_list = []
for i in range(0, len(hosts_iNat)):
    column_list = column_list + [i.columns]
print(column_list)